# Visualizer adapted for Deep Dream

In [1]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from PIL import Image
from fastai.vision.all import *

In [2]:
from settings import categories
image_categories = {}
for axis in categories:
    image_categories[axis] = list(categories[axis].keys())
image_categories

{'depth': ['iconic', 'symbolic'], 'breath': ['abstract', 'concrete']}

## Load models

In [6]:
# Load a pretrained/pretuned model to visually analyse
model_name = "googlenet"
model_root = f"models/{model_name}_10x"
model_focus = list(categories.keys())
learners = {}
for focus in model_focus:
    learners[focus] = load_learner(f"{model_root}/{focus}.pkl")

/home/trpquo/miniforge3/envs/fastai/lib/python3.12/site-packages/fastai/learner.py:455: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")


In [ ]:
device = next(iter(learners.values())).model.parameters().__next__().device

In [7]:
for focus, learn in learners.items():
    print(focus, list(enumerate(learn.dls.vocab)))

depth [(0, 'iconic'), (1, 'symbolic')]
breath [(0, 'abstract'), (1, 'concrete')]


In [8]:
def get_inception_blocks(model):
    blocks = []

    for name, module in model.named_modules():
        if name == "":
            continue

        class_name = module.__class__.__name__.lower()

        if "inception" in class_name and len(list(module.children())) >= 2:
            blocks.append((name, module))

    return blocks

In [18]:
module_names = {
    "0.5": "Inception3a",
    "0.6": "Inception3b",
    "0.8": "Inception4a",
    "0.9": "Inception4b",
    "0.10": "Inception4c",
    "0.11": "Inception4d",
    "0.12": "Inception4e",
    "0.14": "Inception5a",
    "0.15": "Inception5b",
}
for focus, learn in learners.items():
    blocks = get_inception_blocks(learn.model)

    print(f"\n{focus}")
    for i, (name, module) in enumerate(blocks):
        print(f"{i:02d} -> {name}: {module_names[name]}")


depth
00 -> 0.5: Inception3a
01 -> 0.6: Inception3b
02 -> 0.8: Inception4a
03 -> 0.9: Inception4b
04 -> 0.10: Inception4c
05 -> 0.11: Inception4d
06 -> 0.12: Inception4e
07 -> 0.14: Inception5a
08 -> 0.15: Inception5b

breath
00 -> 0.5: Inception3a
01 -> 0.6: Inception3b
02 -> 0.8: Inception4a
03 -> 0.9: Inception4b
04 -> 0.10: Inception4c
05 -> 0.11: Inception4d
06 -> 0.12: Inception4e
07 -> 0.14: Inception5a
08 -> 0.15: Inception5b


In [19]:
def preprocess_for_model(x_rgb, learn):
    """
    x_rgb: N x 3 x H x W tensor in [0, 1].
    Returns the tensor expected by learn.model.
    """
    x = x_rgb

    for transform in learn.dls.after_batch.fs:
        x = transform(x)

    return x

In [20]:
for focus, learn in learners.items():
    print(f"\n{focus} after_batch:")
    print(learn.dls.after_batch)


depth after_batch:
Pipeline: IntToFloatTensor -- {'div': 255.0, 'div_mask': 1}

breath after_batch:
Pipeline: IntToFloatTensor -- {'div': 255.0, 'div_mask': 1}


### Helper functions

In [21]:
def total_variation(x):
    horizontal = (x[:, :, :, 1:] - x[:, :, :, :-1]).abs().mean()
    vertical = (x[:, :, 1:, :] - x[:, :, :-1, :]).abs().mean()
    return horizontal + vertical


def save_rgb_tensor(x, path):
    """
    x: 1 x 3 x H x W tensor in [0, 1].
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    img = (
        x.detach()
        .cpu()
        .squeeze(0)
        .permute(1, 2, 0)
        .clamp(0, 1)
        .numpy()
    )

    Image.fromarray((img * 255).round().astype(np.uint8)).save(path)


def numpy_to_rgb_tensor(img, device):
    """
    img: H x W x 3 NumPy array in [0, 1].
    """
    return (
        torch.from_numpy(img)
        .float()
        .permute(2, 0, 1)
        .unsqueeze(0)
        .to(device)
        .clamp(0, 1)
    )


def output_index(learn, class_name):
    vocab = list(learn.dls.vocab)

    if class_name not in vocab:
        raise ValueError(
            f"{class_name!r} not found in learner vocabulary {vocab}"
        )

    return vocab.index(class_name)


def get_class_logit(logits, class_index):
    """
    Both of your models return shape [N, 2].
    """
    if logits.ndim != 2 or logits.shape[1] != 2:
        raise ValueError(
            f"Expected logits with shape [N, 2], got {tuple(logits.shape)}"
        )

    return logits[:, class_index].mean()

## Activation Maximization
### Select channels by mean activation

In [27]:
def get_module_by_name(model, name):
    for module_name, module in model.named_modules():
        if module_name == name:
            return module

    raise KeyError(f"Layer not found: {name}")


def top_channels_by_mean_activation(
    model,
    layer_name,
    img_tensor,
    top_n=4,
):
    layer = get_module_by_name(model, layer_name)
    activations = {}

    def hook_fn(module, inputs, output):
        activations["value"] = output.detach()

    hook = layer.register_forward_hook(hook_fn)

    model.eval()
    with torch.no_grad():
        _ = model(img_tensor)

    hook.remove()

    a = activations["value"]

    if a.ndim != 4:
        raise ValueError(
            f"Expected N x C x H x W activation, got {tuple(a.shape)}"
        )

    channel_means = a.mean(dim=(0, 2, 3))
    values, indices = torch.topk(
        channel_means,
        k=min(top_n, channel_means.numel()),
    )

    return indices.tolist(), values.tolist()

In [28]:
def get_image_paths(folder):
    extensions = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}
    return [
        p for p in Path(folder).glob("*")
        if p.suffix.lower() in extensions
    ]


def random_image_path(folder):
    paths = get_image_paths(folder)

    if not paths:
        raise ValueError(f"No images found in {folder}")

    return random.choice(paths)


def image_to_model_tensor(path, learn, device):
    img = PILImage.create(path)
    dl = learn.dls.test_dl([img])
    xb = next(iter(dl))[0].to(device)
    return xb

In [29]:
selected_channel_records = []

for focus, learn in learners.items():
    blocks = get_inception_blocks(learn.model)

    folder = Path("artefacts") / focus
    category_a, category_b = image_categories[focus]

    path_a = random_image_path(folder / category_a)
    path_b = random_image_path(folder / category_b)

    img_a = image_to_model_tensor(path_a, learn, device)
    img_b = image_to_model_tensor(path_b, learn, device)

    print(f"\n{focus}")
    print("A:", path_a)
    print("B:", path_b)

    for block_index, (layer_name, module) in enumerate(blocks):
        channels_a, scores_a = top_channels_by_mean_activation(
            learn.model,
            layer_name,
            img_a,
            top_n=4,
        )

        channels_b, scores_b = top_channels_by_mean_activation(
            learn.model,
            layer_name,
            img_b,
            top_n=4,
        )

        selected_channels = sorted(set(channels_a + channels_b))

        print(
            f"{layer_name}: "
            f"A={channels_a}, B={channels_b}, "
            f"selected={selected_channels}"
        )

        for channel in selected_channels:
            selected_channel_records.append({
                "focus": focus,
                "block_index": block_index,
                "layer_name": layer_name,
                "channel": channel,
                "source_a": str(path_a),
                "source_b": str(path_b),
            })


depth
A: artefacts/depth/iconic/9223372032559864316.jpg
B: artefacts/depth/symbolic/5402.jpg
0.5: A=[198, 52, 19, 39], B=[39, 52, 198, 15], selected=[15, 19, 39, 52, 198]
0.6: A=[29, 106, 50, 80], B=[52, 29, 19, 73], selected=[19, 29, 50, 52, 73, 80, 106]
0.8: A=[415, 498, 14, 463], B=[415, 157, 498, 489], selected=[14, 157, 415, 463, 489, 498]
0.9: A=[85, 137, 77, 76], B=[85, 75, 154, 137], selected=[75, 76, 77, 85, 137, 154]
0.10: A=[20, 31, 86, 126], B=[20, 31, 86, 126], selected=[20, 31, 86, 126]
0.11: A=[62, 460, 42, 432], B=[62, 90, 91, 460], selected=[42, 62, 90, 91, 432, 460]
0.12: A=[790, 150, 828, 742], B=[724, 687, 799, 674], selected=[150, 674, 687, 724, 742, 790, 799, 828]
0.14: A=[797, 365, 788, 357], B=[743, 381, 774, 403], selected=[357, 365, 381, 403, 743, 774, 788, 797]
0.15: A=[933, 489, 925, 562], B=[289, 517, 172, 859], selected=[172, 289, 489, 517, 562, 859, 925, 933]

breath
A: artefacts/breath/abstract/289163.jpg
B: artefacts/breath/concrete/14547.jpg
0.5: A=[1

### Channel activation maximization

In [25]:
def optimize_channel_rgb(
    model,
    learn,
    layer_name,
    channel,
    img_size=224,
    steps=60,
    lr=0.08,
    tv_weight=1e-5,
    l2_weight=1e-6,
    jitter=8,
    device=None,
):
    model.eval()

    if device is None:
        device = next(model.parameters()).device

    layer = get_module_by_name(model, layer_name)
    activations = {}

    def hook_fn(module, inputs, output):
        activations["value"] = output

    hook = layer.register_forward_hook(hook_fn)

    x = torch.rand(
        1,
        3,
        img_size,
        img_size,
        device=device,
        requires_grad=True,
    )

    optimizer = torch.optim.Adam([x], lr=lr)

    for _ in range(steps):
        optimizer.zero_grad(set_to_none=True)

        if jitter > 0:
            ox = int(torch.randint(
                -jitter,
                jitter + 1,
                (1,),
                device=device,
            ).item())

            oy = int(torch.randint(
                -jitter,
                jitter + 1,
                (1,),
                device=device,
            ).item())

            x_in = torch.roll(x, shifts=(ox, oy), dims=(2, 3))
        else:
            x_in = x

        x_model = preprocess_for_model(x_in, learn)
        _ = model(x_model)

        a = activations["value"]

        if a.ndim != 4:
            raise ValueError(
                f"Expected N x C x H x W activation, got {tuple(a.shape)}"
            )

        channel_activation = a[:, channel].mean()
        tv = total_variation(x)
        l2 = x.pow(2).mean()

        objective = (
            channel_activation
            - tv_weight * tv
            - l2_weight * l2
        )

        (-objective).backward()
        optimizer.step()

        with torch.no_grad():
            x.clamp_(0, 1)

    hook.remove()

    return x.detach()

### Generate activation-maximization visualizations

In [26]:
prototype_root = Path("visualizations") / model_name / "channel_prototypes"
prototype_root.mkdir(parents=True, exist_ok=True)

prototype_records = []

for record in selected_channel_records:
    focus = record["focus"]
    layer_name = record["layer_name"]
    channel = record["channel"]
    learn = learners[focus]

    safe_layer_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", module_names[layer_name])

    filename = (
        f"{safe_layer_name}-ch{channel:03d}.png"
    )

    save_path = prototype_root / focus / filename

    print(
        f"Generating prototype: "
        f"{focus} / {layer_name} / channel {channel}"
    )

    x_proto = optimize_channel_rgb(
        model=learn.model,
        learn=learn,
        layer_name=layer_name,
        channel=channel,
        img_size=224,
        steps=60,
        lr=0.08,
        tv_weight=1e-5,
        l2_weight=1e-6,
        jitter=8,
        device=device,
    )

    save_rgb_tensor(x_proto, save_path)

    prototype_records.append({
        **record,
        "prototype_path": str(save_path),
    })

prototype_df = pd.DataFrame(prototype_records)
prototype_df

Generating prototype: depth / 0.5 / channel 23
Generating prototype: depth / 0.5 / channel 39
Generating prototype: depth / 0.5 / channel 52
Generating prototype: depth / 0.5 / channel 198
Generating prototype: depth / 0.6 / channel 19
Generating prototype: depth / 0.6 / channel 29
Generating prototype: depth / 0.6 / channel 50
Generating prototype: depth / 0.6 / channel 52
Generating prototype: depth / 0.6 / channel 73
Generating prototype: depth / 0.6 / channel 465
Generating prototype: depth / 0.8 / channel 44
Generating prototype: depth / 0.8 / channel 55
Generating prototype: depth / 0.8 / channel 156
Generating prototype: depth / 0.8 / channel 415
Generating prototype: depth / 0.8 / channel 462
Generating prototype: depth / 0.8 / channel 469
Generating prototype: depth / 0.8 / channel 489
Generating prototype: depth / 0.9 / channel 75
Generating prototype: depth / 0.9 / channel 76
Generating prototype: depth / 0.9 / channel 77
Generating prototype: depth / 0.9 / channel 85
Genera

,focus,block_index,layer_name,channel,source_a,source_b,prototype_path
0,depth,0,0.5,23,artefacts/depth/iconic/11030.jpg,artefacts/depth/symbolic/225188.jpg,visualizations/googlenet/channel_prototypes/depth/Inception3a-ch023.png
1,depth,0,0.5,39,artefacts/depth/iconic/11030.jpg,artefacts/depth/symbolic/225188.jpg,visualizations/googlenet/channel_prototypes/depth/Inception3a-ch039.png
2,depth,0,0.5,52,artefacts/depth/iconic/11030.jpg,artefacts/depth/symbolic/225188.jpg,visualizations/googlenet/channel_prototypes/depth/Inception3a-ch052.png
3,depth,0,0.5,198,artefacts/depth/iconic/11030.jpg,artefacts/depth/symbolic/225188.jpg,visualizations/googlenet/channel_prototypes/depth/Inception3a-ch198.png
4,depth,1,0.6,19,artefacts/depth/iconic/11030.jpg,artefacts/depth/symbolic/225188.jpg,visualizations/googlenet/channel_prototypes/depth/Inception3b-ch019.png
...,...,...,...,...,...,...,...
114,breath,8,0.15,717,artefacts/breath/abstract/22964.jpg,artefacts/breath/concrete/20713.jpg,visualizations/googlenet/channel_prototypes/breath/Inception5b-ch717.png
115,breath,8,0.15,772,artefacts/breath/abstract/22964.jpg,artefacts/breath/concrete/20713.jpg,visualizations/googlenet/channel_prototypes/breath/Inception5b-ch772.png
116,breath,8,0.15,805,artefacts/breath/abstract/22964.jpg,artefacts/breath/concrete/20713.jpg,visualizations/googlenet/channel_prototypes/breath/Inception5b-ch805.png
117,breath,8,0.15,842,artefacts/breath/abstract/22964.jpg,artefacts/breath/concrete/20713.jpg,visualizations/googlenet/channel_prototypes/breath/Inception5b-ch842.png


## Class-directed DeepDream objective

In [30]:
def optimize_class_deepdream(
    model,
    learn,
    seed_path,
    target_class,
    layer_name,
    channel,
    steps=100,
    lr=0.01,
    class_weight=1.0,
    channel_weight=0.2,
    tv_weight=1e-5,
    l2_weight=1e-6,
    jitter=8,
    normalize_gradient=True,
    device=None,
):
    model.eval()

    if device is None:
        device = next(model.parameters()).device

    seed_img = np.asarray(Image.open(seed_path).convert("RGB"))
    seed_img = seed_img.astype(np.float32) / 255.0

    x = numpy_to_rgb_tensor(seed_img, device)
    x.requires_grad_(True)

    target_index = output_index(learn, target_class)

    target_layer = get_module_by_name(model, layer_name)
    activations = {}

    def hook_fn(module, inputs, output):
        activations["value"] = output

    hook = target_layer.register_forward_hook(hook_fn)

    optimizer = torch.optim.Adam([x], lr=lr)

    history = []

    for step in range(steps):
        optimizer.zero_grad(set_to_none=True)

        if jitter > 0:
            ox = int(torch.randint(
                -jitter,
                jitter + 1,
                (1,),
                device=device,
            ).item())

            oy = int(torch.randint(
                -jitter,
                jitter + 1,
                (1,),
                device=device,
            ).item())

            x_in = torch.roll(x, shifts=(ox, oy), dims=(2, 3))
        else:
            x_in = x

        x_model = preprocess_for_model(x_in, learn)
        logits = model(x_model)

        class_logit = get_class_logit(logits, target_index)

        a = activations["value"]

        if a.ndim != 4:
            raise ValueError(
                f"Expected N x C x H x W activation, got {tuple(a.shape)}"
            )

        channel_activation = a[:, channel].mean()

        tv = total_variation(x)
        l2 = x.pow(2).mean()

        objective = (
            class_weight * class_logit
            + channel_weight * channel_activation
            - tv_weight * tv
            - l2_weight * l2
        )

        (-objective).backward()

        if normalize_gradient:
            with torch.no_grad():
                grad = x.grad
                grad_norm = grad.abs().mean().clamp_min(1e-8)
                x.grad.div_(grad_norm)

        optimizer.step()

        with torch.no_grad():
            x.clamp_(0, 1)

        with torch.no_grad():
            probabilities = logits.softmax(dim=1)
            history.append({
                "step": step,
                "objective": float(objective.detach().cpu()),
                "class_logit": float(class_logit.detach().cpu()),
                "channel_activation": float(
                    channel_activation.detach().cpu()
                ),
                "probability": float(
                    probabilities[:, target_index].detach().cpu()
                ),
                "tv": float(tv.detach().cpu()),
                "l2": float(l2.detach().cpu()),
            })

    hook.remove()

    return x.detach(), pd.DataFrame(history)

In [ ]:
target_classes = {
    "depth": "symbolic",
    "breath": "concrete",
    # "depth": "iconic",
    # "breath": "abstract",
}

In [31]:
dd_root = Path("visualizations") / model_name / "deepdream"
dd_root.mkdir(parents=True, exist_ok=True)

dd_records = []

for record in prototype_records:
    focus = record["focus"]
    layer_name = record["layer_name"]
    channel = record["channel"]
    prototype_path = record["prototype_path"]

    learn = learners[focus]
    target_class = target_classes[focus]

    safe_layer_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", module_names[layer_name])

    filename = (
        f"dd-{safe_layer_name}-ch{channel:03d}-{target_class}.png"
    )

    save_path = dd_root / focus / filename

    print(
        f"DeepDream: {focus} / {layer_name} / "
        f"channel {channel} / class {target_class}"
    )

    x_dd, history = optimize_class_deepdream(
        model=learn.model,
        learn=learn,
        seed_path=prototype_path,
        target_class=target_class,
        layer_name=layer_name,
        channel=channel,
        steps=100,
        lr=0.01,
        class_weight=1.0,
        channel_weight=0.2,
        tv_weight=1e-5,
        l2_weight=1e-6,
        jitter=8,
        normalize_gradient=True,
        device=device,
    )

    save_rgb_tensor(x_dd, save_path)

    dd_records.append({
        **record,
        "target_class": target_class,
        "dd_path": str(save_path),
        "final_class_logit": history.iloc[-1]["class_logit"],
        "final_probability": history.iloc[-1]["probability"],
        "final_channel_activation": (
            history.iloc[-1]["channel_activation"]
        ),
    })

dd_df = pd.DataFrame(dd_records)
dd_df

DeepDream: depth / 0.5 / channel 23 / class symbolic
DeepDream: depth / 0.5 / channel 39 / class symbolic
DeepDream: depth / 0.5 / channel 52 / class symbolic
DeepDream: depth / 0.5 / channel 198 / class symbolic
DeepDream: depth / 0.6 / channel 19 / class symbolic
DeepDream: depth / 0.6 / channel 29 / class symbolic
DeepDream: depth / 0.6 / channel 50 / class symbolic
DeepDream: depth / 0.6 / channel 52 / class symbolic
DeepDream: depth / 0.6 / channel 73 / class symbolic
DeepDream: depth / 0.6 / channel 465 / class symbolic
DeepDream: depth / 0.8 / channel 44 / class symbolic
DeepDream: depth / 0.8 / channel 55 / class symbolic
DeepDream: depth / 0.8 / channel 156 / class symbolic
DeepDream: depth / 0.8 / channel 415 / class symbolic
DeepDream: depth / 0.8 / channel 462 / class symbolic
DeepDream: depth / 0.8 / channel 469 / class symbolic
DeepDream: depth / 0.8 / channel 489 / class symbolic
DeepDream: depth / 0.9 / channel 75 / class symbolic
DeepDream: depth / 0.9 / channel 76 / c

,focus,block_index,layer_name,channel,source_a,source_b,prototype_path,target_class,dd_path,final_class_logit,final_probability,final_channel_activation
0,depth,0,0.5,23,artefacts/depth/iconic/11030.jpg,artefacts/depth/symbolic/225188.jpg,visualizations/googlenet/channel_prototypes/depth/Inception3a-ch023.png,symbolic,visualizations/googlenet/deepdream/depth/dd-Inception3a-ch023-symbolic.png,12.681081,1.0,18.110134
1,depth,0,0.5,39,artefacts/depth/iconic/11030.jpg,artefacts/depth/symbolic/225188.jpg,visualizations/googlenet/channel_prototypes/depth/Inception3a-ch039.png,symbolic,visualizations/googlenet/deepdream/depth/dd-Inception3a-ch039-symbolic.png,11.057767,1.0,27.658211
2,depth,0,0.5,52,artefacts/depth/iconic/11030.jpg,artefacts/depth/symbolic/225188.jpg,visualizations/googlenet/channel_prototypes/depth/Inception3a-ch052.png,symbolic,visualizations/googlenet/deepdream/depth/dd-Inception3a-ch052-symbolic.png,24.645441,1.0,11.371732
3,depth,0,0.5,198,artefacts/depth/iconic/11030.jpg,artefacts/depth/symbolic/225188.jpg,visualizations/googlenet/channel_prototypes/depth/Inception3a-ch198.png,symbolic,visualizations/googlenet/deepdream/depth/dd-Inception3a-ch198-symbolic.png,18.823912,1.0,6.933724
4,depth,1,0.6,19,artefacts/depth/iconic/11030.jpg,artefacts/depth/symbolic/225188.jpg,visualizations/googlenet/channel_prototypes/depth/Inception3b-ch019.png,symbolic,visualizations/googlenet/deepdream/depth/dd-Inception3b-ch019-symbolic.png,37.026424,1.0,0.210827
...,...,...,...,...,...,...,...,...,...,...,...,...
114,breath,8,0.15,717,artefacts/breath/abstract/22964.jpg,artefacts/breath/concrete/20713.jpg,visualizations/googlenet/channel_prototypes/breath/Inception5b-ch717.png,concrete,visualizations/googlenet/deepdream/breath/dd-Inception5b-ch717-concrete.png,28.118706,1.0,3.092068
115,breath,8,0.15,772,artefacts/breath/abstract/22964.jpg,artefacts/breath/concrete/20713.jpg,visualizations/googlenet/channel_prototypes/breath/Inception5b-ch772.png,concrete,visualizations/googlenet/deepdream/breath/dd-Inception5b-ch772-concrete.png,30.121323,1.0,3.647279
116,breath,8,0.15,805,artefacts/breath/abstract/22964.jpg,artefacts/breath/concrete/20713.jpg,visualizations/googlenet/channel_prototypes/breath/Inception5b-ch805.png,concrete,visualizations/googlenet/deepdream/breath/dd-Inception5b-ch805-concrete.png,29.149126,1.0,6.790474
117,breath,8,0.15,842,artefacts/breath/abstract/22964.jpg,artefacts/breath/concrete/20713.jpg,visualizations/googlenet/channel_prototypes/breath/Inception5b-ch842.png,concrete,visualizations/googlenet/deepdream/breath/dd-Inception5b-ch842-concrete.png,24.656712,1.0,1.702438


In [ ]:
def load_display_image(path):
    return np.asarray(Image.open(path).convert("RGB")) / 255.0

for _, row in dd_df.iterrows():
    prototype = load_display_image(row["prototype_path"])
    dream = load_display_image(row["dd_path"])

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))

    axes[0].imshow(prototype)
    axes[0].set_title(
        f"Prototype\n{row['layer_name']} ch{row['channel']}"
    )
    axes[0].axis("off")

    axes[1].imshow(dream)
    axes[1].set_title(
        f"DeepDream\n{row['target_class']}"
    )
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

### Validate that the class objective worked

In [32]:
def classify_saved_image(path, learn, device):
    img = PILImage.create(path)
    xb = next(iter(learn.dls.test_dl([img])))[0].to(device)

    learn.model.eval()

    with torch.no_grad():
        logits = learn.model(xb)
        probs = logits.softmax(dim=1)

    return (
        logits.squeeze(0).cpu(),
        probs.squeeze(0).cpu(),
        list(learn.dls.vocab),
    )

In [33]:
for _, row in dd_df.iterrows():
    learn = learners[row["focus"]]

    logits, probs, vocab = classify_saved_image(
        row["dd_path"],
        learn,
        device,
    )

    print(
        row["dd_path"],
        "\n logits:", dict(zip(vocab, logits.tolist())),
        "\n probs: ", dict(zip(vocab, probs.tolist())),
        "\n_________________________________________________\n"
    )

visualizations/googlenet/deepdream/depth/dd-Inception3a-ch023-symbolic.png 
 logits: {'iconic': 0.18434187769889832, 'symbolic': 2.882518768310547} 
 probs:  {'iconic': 0.06308102607727051, 'symbolic': 0.9369190335273743} 
_________________________________________________

visualizations/googlenet/deepdream/depth/dd-Inception3a-ch039-symbolic.png 
 logits: {'iconic': 1.0541235208511353, 'symbolic': 0.7425670027732849} 
 probs:  {'iconic': 0.5772651433944702, 'symbolic': 0.4227348566055298} 
_________________________________________________

visualizations/googlenet/deepdream/depth/dd-Inception3a-ch052-symbolic.png 
 logits: {'iconic': 1.3809212446212769, 'symbolic': 1.9484390020370483} 
 probs:  {'iconic': 0.36180979013442993, 'symbolic': 0.6381902098655701} 
_________________________________________________

visualizations/googlenet/deepdream/depth/dd-Inception3a-ch198-symbolic.png 
 logits: {'iconic': -3.9407906532287598, 'symbolic': 6.2954301834106445} 
 probs:  {'iconic': 3.5846765

In [34]:
def evaluate_image(path, model, learn, layer_name, channel, device):
    img = PILImage.create(path)
    xb = next(iter(learn.dls.test_dl([img])))[0].to(device)

    layer = get_module_by_name(model, layer_name)
    acts = {}

    def hook_fn(module, inputs, output):
        acts["value"] = output.detach()

    hook = layer.register_forward_hook(hook_fn)

    model.eval()
    with torch.no_grad():
        logits = model(xb)
        activation = acts["value"][:, channel].mean()

    hook.remove()

    probs = logits.softmax(dim=1)
    return {
        "logit_0": logits[0, 0].item(),
        "logit_1": logits[0, 1].item(),
        "prob_0": probs[0, 0].item(),
        "prob_1": probs[0, 1].item(),
        "channel_activation": activation.item(),
    }

In [35]:
rows = []

for _, row in dd_df.iterrows():
    model = learners[row["focus"]].model
    learn = learners[row["focus"]]

    before = evaluate_image(
        row["prototype_path"],
        model,
        learn,
        row["layer_name"],
        row["channel"],
        device,
    )

    after = evaluate_image(
        row["dd_path"],
        model,
        learn,
        row["layer_name"],
        row["channel"],
        device,
    )

    rows.append({
        **row.to_dict(),
        **{f"before_{k}": v for k, v in before.items()},
        **{f"after_{k}": v for k, v in after.items()},
    })

comparison_df = pd.DataFrame(rows)

comparison_df[
    [
        "focus",
        "layer_name",
        "channel",
        "target_class",
        "before_logit_0",
        "before_logit_1",
        "after_logit_0",
        "after_logit_1",
        "before_channel_activation",
        "after_channel_activation",
    ]
].head()

,focus,layer_name,channel,target_class,before_logit_0,before_logit_1,after_logit_0,after_logit_1,before_channel_activation,after_channel_activation
0,depth,0.5,23,symbolic,2.422901,-0.032270,0.184342,2.882519,16.193285,13.389845
1,depth,0.5,39,symbolic,2.369991,-0.456281,1.054124,0.742567,14.969568,13.118427
2,depth,0.5,52,symbolic,3.568006,-1.741208,1.380921,1.948439,10.684602,8.115097
3,depth,0.5,198,symbolic,-0.426601,3.075995,-3.940791,6.295430,5.280715,4.787465
4,depth,0.6,19,symbolic,1.521085,-0.571726,-6.777217,11.531037,0.405200,0.283759


In [38]:
comparison_df["before_margin_1_minus_0"] = (
    comparison_df["before_logit_1"]
    - comparison_df["before_logit_0"]
)

comparison_df["after_margin_1_minus_0"] = (
    comparison_df["after_logit_1"]
    - comparison_df["after_logit_0"]
)

comparison_df

,focus,block_index,layer_name,channel,source_a,source_b,prototype_path,target_class,dd_path,final_class_logit,...,before_prob_0,before_prob_1,before_channel_activation,after_logit_0,after_logit_1,after_prob_0,after_prob_1,after_channel_activation,before_margin_1_minus_0,after_margin_1_minus_0
0,depth,0,0.5,23,artefacts/depth/iconic/11030.jpg,artefacts/depth/symbolic/225188.jpg,visualizations/googlenet/channel_prototypes/depth/Inception3a-ch023.png,symbolic,visualizations/googlenet/deepdream/depth/dd-Inception3a-ch023-symbolic.png,12.681081,...,0.920939,7.906125e-02,16.193285,0.184342,2.882519,6.308103e-02,0.936919,13.389845,-2.455171,2.698177
1,depth,0,0.5,39,artefacts/depth/iconic/11030.jpg,artefacts/depth/symbolic/225188.jpg,visualizations/googlenet/channel_prototypes/depth/Inception3a-ch039.png,symbolic,visualizations/googlenet/deepdream/depth/dd-Inception3a-ch039-symbolic.png,11.057767,...,0.944079,5.592088e-02,14.969568,1.054124,0.742567,5.772651e-01,0.422735,13.118427,-2.826272,-0.311557
2,depth,0,0.5,52,artefacts/depth/iconic/11030.jpg,artefacts/depth/symbolic/225188.jpg,visualizations/googlenet/channel_prototypes/depth/Inception3a-ch052.png,symbolic,visualizations/googlenet/deepdream/depth/dd-Inception3a-ch052-symbolic.png,24.645441,...,0.995079,4.921469e-03,10.684602,1.380921,1.948439,3.618098e-01,0.638190,8.115097,-5.309215,0.567518
3,depth,0,0.5,198,artefacts/depth/iconic/11030.jpg,artefacts/depth/symbolic/225188.jpg,visualizations/googlenet/channel_prototypes/depth/Inception3a-ch198.png,symbolic,visualizations/googlenet/deepdream/depth/dd-Inception3a-ch198-symbolic.png,18.823912,...,0.029238,9.707616e-01,5.280715,-3.940791,6.295430,3.584677e-05,0.999964,4.787465,3.502596,10.236221
4,depth,1,0.6,19,artefacts/depth/iconic/11030.jpg,artefacts/depth/symbolic/225188.jpg,visualizations/googlenet/channel_prototypes/depth/Inception3b-ch019.png,symbolic,visualizations/googlenet/deepdream/depth/dd-Inception3b-ch019-symbolic.png,37.026424,...,0.890203,1.097975e-01,0.405200,-6.777217,11.531037,1.118990e-08,1.000000,0.283759,-2.092812,18.308255
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114,breath,8,0.15,717,artefacts/breath/abstract/22964.jpg,artefacts/breath/concrete/20713.jpg,visualizations/googlenet/channel_prototypes/breath/Inception5b-ch717.png,concrete,visualizations/googlenet/deepdream/breath/dd-Inception5b-ch717-concrete.png,28.118706,...,0.999996,4.200109e-06,0.902752,1.869366,-2.860528,9.912498e-01,0.008750,0.625592,-12.380396,-4.729894
115,breath,8,0.15,772,artefacts/breath/abstract/22964.jpg,artefacts/breath/concrete/20713.jpg,visualizations/googlenet/channel_prototypes/breath/Inception5b-ch772.png,concrete,visualizations/googlenet/deepdream/breath/dd-Inception5b-ch772-concrete.png,30.121323,...,0.999994,6.128211e-06,0.634741,-0.415730,4.418983,7.886281e-03,0.992114,0.334981,-12.002602,4.834713
116,breath,8,0.15,805,artefacts/breath/abstract/22964.jpg,artefacts/breath/concrete/20713.jpg,visualizations/googlenet/channel_prototypes/breath/Inception5b-ch805.png,concrete,visualizations/googlenet/deepdream/breath/dd-Inception5b-ch805-concrete.png,29.149126,...,0.999986,1.409358e-05,0.531038,1.730180,-2.153848,9.798467e-01,0.020153,0.754276,-11.169777,-3.884028
117,breath,8,0.15,842,artefacts/breath/abstract/22964.jpg,artefacts/breath/concrete/20713.jpg,visualizations/googlenet/channel_prototypes/breath/Inception5b-ch842.png,concrete,visualizations/googlenet/deepdream/breath/dd-Inception5b-ch842-concrete.png,24.656712,...,1.000000,1.472974e-07,0.217497,0.338840,-1.678885,8.826457e-01,0.117354,0.175072,-15.730812,-2.017726
